# 1-1절 연습 문제 풀이

이 노트북은 1-1절 연습 문제(1-1 ~ 1-5)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `code_examples/ch01/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import torch

## 연습 문제 1-1

> 파이토치 프레임워크를 통해 딥러닝 모델 개발에 사용할 수 있는 여러 데이터셋을 쉽게 구할 수 있다.
> 다음 코드는 MNIST 데이터셋을 불러와 `images`와 `labels` 두 텐서를 생성한다. (코드 1-22)
> 이렇게 생성한 `images`와 `labels` 두 텐서의 형태를 출력해 보자.

In [2]:
from torchvision import datasets, transforms

# 원문 코드의 root='./data' 대신 저장소 규약(공통코드컨벤션 13.9)의 download 디렉터리를 사용
transform = transforms.ToTensor()
train_set = datasets.MNIST(root='../../download', train=True, download=True,
                           transform=transform)
train_loader = torch.utils.data.DataLoader(dataset=train_set, batch_size=32,
                                           shuffle=True)
images, labels = next(iter(train_loader))

print(f'images 텐서의 형태: {images.shape}')
print(f'labels 텐서의 형태: {labels.shape}')

images 텐서의 형태: torch.Size([32, 1, 28, 28])
labels 텐서의 형태: torch.Size([32])


**풀이 해설**

`images`는 `(32, 1, 28, 28)`, `labels`는 `(32,)` 형태다.
배치 크기 32가 첫 번째 차원이고, 두 번째 차원 1은 회색조 이미지의 색상 채널,
마지막 두 차원 28, 28이 이미지의 가로세로 픽셀이다.
`labels`는 이미지 32장 각각의 정답 숫자 하나씩이므로 1차원이다.

**문제 검토**

- **적절성: 적합.** 1-1절에서 배운 `shape` 속성을 실제 데이터에 처음 적용해 보는 문제다.
  다음 문제 1-2, 1-3이 이 텐서를 이어받아 `squeeze()`와 `flatten()`으로 확장되므로 세 문제의 연결이 좋다.
- **[검토] `root='./data'` 경로.** 문제의 코드를 그대로 실행하면 독자가 노트북을 연 위치에 `data` 디렉터리가
  새로 생기고 MNIST를 내려받는다. 저장소에는 이미 `data` 디렉터리가 있으므로 같은 데이터가 두 벌이 될 수 있다.
  예제 노트북들이 쓰는 상대 경로와도 어긋난다. `root='../../data'`로 맞추거나, 각주로 한 줄 안내하는 편이 좋다.
- **[검토] 4장 예고와의 관계.** 각주 8이 "MNIST 데이터셋과 사용에 대한 내용은 4장에서 다룬다"로 안내하는데,
  `DataLoader`, `next(iter(...))`, `transform` 모두 1장에서 설명하지 않은 도구다.
  문제 자체는 형태만 출력하면 되므로 풀 수는 있지만, 지문에 "코드의 내용은 4장에서 다루므로 지금은
  실행 결과만 확인하면 된다"는 한 문장이 있으면 독자가 덜 헤맨다.

## 연습 문제 1-2

> [연습 문제 1-1]에서 생성한 텐서 중 `images` 텐서에는 가로, 세로 각각 28픽셀로 구성된 32장의 이미지 정보가
> 저장되어 있다. 이 텐서의 두 번째 차원은 컬러 이미지의 색상별 데이터로 구성된 색상 채널 차원인데,
> MNIST 데이터와 같이 색상 채널이 하나뿐인 회색조 이미지에서는 색상 채널 차원이 불필요하다.
> `images` 텐서에서 색상 채널 차원을 제거해 보자.

In [3]:
# 색상 채널은 두 번째 차원(인덱스 1)이며 크기가 1이므로 squeeze()로 제거
squeezed = images.squeeze(1)
print(f'제거 전: {images.shape}')
print(f'제거 후: {squeezed.shape}')

# dim 인자를 생략해도 같은 결과 (크기가 1인 차원이 색상 채널뿐이므로)
print(f'dim 생략: {images.squeeze().shape}')

제거 전: torch.Size([32, 1, 28, 28])
제거 후: torch.Size([32, 28, 28])
dim 생략: torch.Size([32, 28, 28])


**풀이 해설**

`(32, 1, 28, 28)`에서 크기가 1인 두 번째 차원을 `squeeze(1)`로 제거해 `(32, 28, 28)`을 얻는다.
이 데이터에서는 크기가 1인 차원이 색상 채널뿐이라 `squeeze()`로 인자를 생략해도 결과가 같다.
다만 배치 크기가 1인 경우(`(1, 1, 28, 28)`)에는 배치 차원까지 사라지므로 `dim`을 지정하는 편이 안전하다.

**문제 검토**

- **적절성: 적합.** `squeeze()`를 '크기가 1인 차원 제거'라는 규칙으로만 외우지 않고,
  색상 채널이라는 실제 의미와 연결해 쓰게 한다.
- **[검토] 지문의 '가로, 세로'.** `images`의 마지막 두 차원은 관례상 (세로, 가로), 즉 (행, 열) 순이다.
  MNIST는 28x28 정사각형이라 결과가 달라지지 않지만, 문제 1-3에서 "마지막 두 차원은 각각 가로 픽셀과
  세로 픽셀"이라고 순서를 명시하므로 정확하지 않은 순서가 굳어질 수 있다.
  "가로세로 28픽셀"처럼 순서를 말하지 않는 표현을 권한다.
- **[검토] `dim` 생략 시의 함정.** 배치 크기가 1이면 `squeeze()`가 배치 차원까지 지운다.
  풀이에서 다루기 좋은 지점이라 해설에 넣었다. 지문에 넣을 필요는 없다.

## 연습 문제 1-3

> [연습 문제 1-2]에서 색상 채널을 제거한 텐서의 마지막 두 차원은 각각 가로 픽셀과 세로 픽셀에 해당하는
> 차원이다. 이 두 차원의 데이터를 이어 붙여 각 이미지가 크기 784의 1차원 텐서로 표현되도록
> `images` 텐서를 변형해 보자.

In [4]:
# 첫 번째 차원(이미지 번호)은 그대로 두고, 두 번째 차원부터 끝까지 이어 붙인다
flattened = squeezed.flatten(1, -1)
print(f'평탄화 전: {squeezed.shape}')
print(f'평탄화 후: {flattened.shape}')

# reshape()로도 같은 결과를 얻을 수 있다
print(f'reshape 사용: {squeezed.reshape(32, 784).shape}')

# 색상 채널 제거와 평탄화를 한 번에 처리할 수도 있다
print(f'한 번에 처리: {images.flatten(1, -1).shape}')

평탄화 전: torch.Size([32, 28, 28])
평탄화 후: torch.Size([32, 784])
reshape 사용: torch.Size([32, 784])
한 번에 처리: torch.Size([32, 784])


**풀이 해설**

`flatten(1, -1)`은 두 번째 차원부터 마지막 차원까지를 하나로 이어 붙인다.
28 × 28 = 784이므로 `(32, 28, 28)`이 `(32, 784)`가 된다.
`reshape(32, 784)`로도 같은 결과를 얻지만, 배치 크기를 숫자로 적어야 해서 배치 크기가 바뀌면 함께 고쳐야 한다.
`flatten(1, -1)`은 배치 크기와 무관하게 동작한다.

한편 `images.flatten(1, -1)`처럼 색상 채널이 남아 있는 상태에서 바로 평탄화해도 결과는 `(32, 784)`로 같다.
크기가 1인 차원은 이어 붙여도 요소 수가 늘지 않기 때문이다.

**문제 검토**

- **적절성: 적합.** 4장 이후 완전 연결 계층에 이미지를 넣을 때 반드시 거치는 변형이라 실전성이 높다.
- **[검토] 지문의 '가로 픽셀과 세로 픽셀'.** 문제 1-2와 같은 문제다. 마지막 두 차원은 (세로, 가로) 순이다.
  또 이 문제에서는 어느 쪽이 가로인지가 풀이에 영향을 주지 않으므로, 순서를 특정하지 않는 편이 안전하다.

**윤문안**

> **1-3**. [연습 문제 1-2]에서 색상 채널을 제거한 텐서의 마지막 두 차원은 이미지의 세로 픽셀과 가로 픽셀에
> 해당하는 차원이다. 이 두 차원을 이어 붙여 각 이미지가 크기 784인 1차원 텐서로 표현되도록 텐서를 변형해 보자.

## 연습 문제 1-4

> 다음은 세 명의 학생이 네 과목의 시험을 두 번 치른 후 성적을 정리한 표이다.
>
> | 학생 | 1회차 국어 | 영어 | 수학 | 과학 | 2회차 국어 | 영어 | 수학 | 과학 |
> |---|---|---|---|---|---|---|---|---|
> | A | 55 | 48 | 97 | 97 | 60 | 5 | 82 | 75 |
> | B | 62 | 36 | 92 | 96 | 66 | 47 | 38 | 65 |
> | C | 73 | 20 | 56 | 100 | 49 | 54 | 43 | 49 |
>
> 1. 이 표의 데이터로 학생 - 회차 - 과목 순의 텐서를 만들어 보자.
> 2. 생성한 텐서를 `permute()` 메서드를 사용해 차원이 과목 - 회차 - 학생 순서인 텐서로 변형해 보자.

In [5]:
# 1) 학생(3) - 회차(2) - 과목(4) 순서의 3차원 텐서
scores = torch.tensor([
    [[55, 48, 97, 97], [60, 5, 82, 75]],      # 학생 A의 1회차, 2회차
    [[62, 36, 92, 96], [66, 47, 38, 65]],     # 학생 B의 1회차, 2회차
    [[73, 20, 56, 100], [49, 54, 43, 49]],    # 학생 C의 1회차, 2회차
])
print(f'학생 - 회차 - 과목 텐서의 형태: {scores.shape}')
print(f'B 학생 2회차 수학 점수: {scores[1, 1, 2]}')

학생 - 회차 - 과목 텐서의 형태: torch.Size([3, 2, 4])
B 학생 2회차 수학 점수: 38


In [6]:
# 2) 과목(4) - 회차(2) - 학생(3) 순서로 차원 재배치
# 원래 차원 순서 (학생=0, 회차=1, 과목=2)를 (과목, 회차, 학생) = (2, 1, 0)으로 지정
by_subject = scores.permute(2, 1, 0)
print(f'과목 - 회차 - 학생 텐서의 형태: {by_subject.shape}')
print(f'수학 2회차 세 학생의 점수: {by_subject[2, 1]}')

# 같은 값을 가리키는지 확인
print(f'같은 값인가: {by_subject[2, 1, 1].item() == scores[1, 1, 2].item()}')

과목 - 회차 - 학생 텐서의 형태: torch.Size([4, 2, 3])
수학 2회차 세 학생의 점수: tensor([82, 38, 43])
같은 값인가: True


**풀이 해설**

표를 읽는 순서가 그대로 중첩 리스트의 중첩 순서가 된다.
가장 바깥이 학생, 그 안이 회차, 가장 안쪽이 과목이므로 `(3, 2, 4)` 형태다.

`permute()`에는 **새 텐서의 각 차원이 원래 몇 번 차원이었는지**를 순서대로 적는다.
새 순서가 (과목, 회차, 학생)이고 원래 번호가 과목=2, 회차=1, 학생=0이므로 `permute(2, 1, 0)`이다.
차원 순서만 바뀔 뿐 값은 그대로이므로, 같은 점수를 서로 다른 인덱스로 가리키게 된다.

**문제 검토**

- **적절성: 적합.** 표를 텐서로 옮기는 과정에서 '중첩 순서 = 차원 순서'를 몸으로 익히게 하고,
  `permute()`의 인자가 '새 순서에 들어갈 원래 차원 번호'라는 점을 확인시킨다.
  세 축의 크기가 3, 2, 4로 모두 달라 실수하면 형태에서 바로 드러나는 것도 좋은 설계다.
- **[확인 요청] A 학생 2회차 영어 점수 `5`.** 24개 값 중 이 값만 한 자리 수다(나머지는 20~100).
  50이나 55에서 숫자 하나가 빠진 오타로 보인다. 의도한 값이라면 그대로 두어도 문제 풀이에는 영향이 없다.
- **[검토] 2번 문항의 검산 안내.** 차원을 재배치한 뒤 결과가 맞는지 스스로 확인할 방법이 지문에 없다.
  "재배치 전후로 같은 점수를 가리키는 인덱스를 찾아 확인해 보자"는 한 문장을 덧붙이면
  `permute()`가 값을 바꾸지 않는다는 점까지 확인하게 된다.

## 연습 문제 1-5 [도전 문제]

> 파이토치가 자동으로 수행하는 브로드캐스팅을 사용하지 않고, 3차원 리스트 `[[[1, 2]], [[3, 4]]]`와
> 2차원 리스트 `[[5, 6], [7, 8]]`로 만든 두 텐서의 덧셈 결과를 출력해 보자.
>
> 힌트: 브로드캐스팅 1단계는 차원을 추가하는 메서드를 사용하면 되며,
> 브로드캐스팅 2단계는 여러 텐서를 이어 붙이는 메서드를 사용하면 된다.

In [7]:
A = torch.tensor([[[1, 2]], [[3, 4]]])    # (2, 1, 2) 형태
B = torch.tensor([[5, 6], [7, 8]])        # (2, 2) 형태
print(f'A: {A.shape}, B: {B.shape}')

# 1단계: 차원 수가 적은 B의 앞쪽에 크기가 1인 차원을 추가해 차원 수를 맞춘다
B_expanded = B.unsqueeze(0)               # (1, 2, 2)
print(f'1단계 후 B: {B_expanded.shape}')

# 2단계: 크기가 1인 차원을 상대 텐서의 크기에 맞춰 복사해 늘린다
#   A는 두 번째 차원(크기 1)을 2로, B는 첫 번째 차원(크기 1)을 2로
A_expanded = torch.cat([A, A], dim=1)                        # (2, 2, 2)
B_expanded = torch.cat([B_expanded, B_expanded], dim=0)      # (2, 2, 2)
print(f'2단계 후 A: {A_expanded.shape}, B: {B_expanded.shape}')

# 이제 형태가 같으므로 브로드캐스팅 없이 요소별 덧셈이 이루어진다
result = A_expanded + B_expanded
print(result)

# 파이토치의 브로드캐스팅 결과와 같은지 확인
print(f'브로드캐스팅 결과와 같은가: {torch.equal(result, A + B)}')

A: torch.Size([2, 1, 2]), B: torch.Size([2, 2])
1단계 후 B: torch.Size([1, 2, 2])
2단계 후 A: torch.Size([2, 2, 2]), B: torch.Size([2, 2, 2])
tensor([[[ 6,  8],
         [ 8, 10]],

        [[ 8, 10],
         [10, 12]]])
브로드캐스팅 결과와 같은가: True


**풀이 해설**

본문이 설명한 브로드캐스팅 2단계를 손으로 재현하는 문제다.

1단계에서 `unsqueeze(0)`으로 `B`를 `(2, 2)`에서 `(1, 2, 2)`로 만들어 차원 수를 3으로 맞춘다.
2단계에서는 크기가 1인 차원을 상대 텐서의 크기만큼 늘려야 하는데, `cat()`으로 자기 자신을 이어 붙이면 된다.
`A`는 두 번째 차원이, `B`는 첫 번째 차원이 각각 크기 1이므로 이어 붙이는 축이 서로 다르다.
두 텐서가 모두 `(2, 2, 2)`가 되면 형태가 같아져 브로드캐스팅 없이 요소별 덧셈이 이루어진다.

`torch.equal()`로 파이토치가 자동으로 계산한 `A + B`와 같은 결과임을 확인할 수 있다.

**문제 검토**

- **적절성: 적합. 도전 문제로 잘 설계됐다.** 브로드캐스팅을 '알아서 되는 것'으로 넘기지 않고,
  본문의 2단계 설명을 그대로 손으로 밟게 만든다. 특히 두 텐서의 확장 축이 서로 다르다는 점을
  직접 겪어야 풀리므로, 본문만 읽고 지나친 독자는 여기서 막히고 다시 읽게 된다.
- **[검토] 예시 텐서가 본문과 같다.** 본문 p10의 브로드캐스팅 설명과 코드 1-9가 똑같은 `(2, 1, 2)`와 `(2, 2)`를
  쓴다. 답을 본문에서 확인할 수 있어 검산이 쉬운 장점이 있는 반면, 새로 생각할 거리는 줄어든다.
  의도한 것으로 보이며 도전 문제의 난도를 고려하면 그대로 두는 편이 낫다고 본다.
- **[검토] 검산 방법 안내.** `torch.equal(result, A + B)`로 스스로 채점할 수 있다는 힌트를 한 줄 덧붙이면
  독자가 답을 확신할 수 있다.